# Penjelasan Jupyter Notebook: Analisis Keparahan Komplain dan Segmentasi

Dokumen ini memuat penjelasan detail untuk skrip yang terdapat di dalam Jupyter Notebook `nlp_sentiment.ipynb`. Berbeda dari pendekatan awal yang mungkin menggunakan pemrosesan bahasa alami (NLP), notebook ini menerapkan pendekatan berbasis aturan (*rule-based*) untuk menilai tingkat keparahan tiket keluhan pelanggan dan melakukan segmentasi perilaku dengan lebih akurat.

## 1: Pemrosesan Tiket, Penilaian Keparahan, dan Segmentasi
Di dalam notebook ini menjalankan keseluruhan alur ekstraksi fitur komplain dengan tahapan-tahapan logis sebagai berikut:

* **Pemuatan dan Penyaringan Data (Anti-Kebocoran):** Sistem membaca data dari `support_tickets.csv` dan `customer_accounts.csv`. Algoritma kemudian mencocokkan tanggal pembuatan tiket (`created_date`) dengan tanggal pembatalan akun (`unsubscribed_date`). Sistem secara ketat membuang tiket komplain yang tercatat *setelah* pelanggan berhenti berlangganan, sehingga model Machine Learning terhindar dari bias masa depan (*data leakage*).
* **Perhitungan Skor Keparahan (Severity Score):** Karena analisis sentimen otomatis dirasa kurang tepat membaca konteks, sistem beralih menggunakan fungsi matematika `hitung_skor_keparahan`. Tiket diberi bobot berdasarkan tingkat prioritasnya: *High/Critical* bernilai 3, *Medium* bernilai 2, dan *Low* bernilai 1. Selain itu, tiket yang masih berstatus *Open* (belum terselesaikan) akan diberikan hukuman tambahan skor (+2) karena berpotensi membuat pelanggan frustrasi.
* **Agregasi Tingkat Pelanggan:** Data tiket diringkas berdasarkan ID Pelanggan (`customer_id`). Proses ini menghitung total tiket yang pernah dibuat, rata-rata skor keparahan, serta melacak secara spesifik berapa banyak tiket yang tergolong "parah" (memiliki skor di atas atau sama dengan 4).
* **Perhitungan Rasio dan Segmentasi Pelanggan:** Skrip menciptakan fitur prediktif penting bernama `severe_ticket_ratio` (persentase keluhan parah). Berdasarkan metrik komplain tersebut, fungsi `segmentasi_tipe_pelanggan` mengelompokkan pelanggan ke dalam 4 tipe:
    * `loyal_quiet`: Pelanggan yang diam dan tidak pernah membuat tiket keluhan.
    * `problematic`: Pelanggan yang memiliki komplain berskala darurat atau mayoritas tiketnya bermasalah/terbengkalai.
    * `at_risk`: Pelanggan yang cukup sering mengeluh atau memiliki beberapa tiket berisiko tinggi.
    * `satisfied`: Pelanggan yang hanya memiliki keluhan ringan dan tiketnya tertangani dengan baik.
* **Penggabungan dan Penanganan Nilai Kosong:** Hasil agregasi digabungkan kembali secara utuh ke master data agar komposisi 3000 akun pelanggan tetap konsisten. Pengguna tanpa riwayat keluhan sama sekali otomatis ditandai sebagai tipe `loyal_quiet`, dan metrik komplain numeriknya diisi dengan nilai 0.
* **Ekspor dan Pelaporan Analitik:** Tabel akhir diekspor menjadi berkas siap pakai `hasil_komplain.csv`. Skrip diakhiri dengan fungsi cetak (*print*) yang mendemonstrasikan rangkuman dasbor analitik sederhana, menampilkan jumlah serta persentase distribusi dari keempat profil tipe pelanggan tersebut.

In [ ]:
import pandas as pd

# 1. loading data & filter anti-bocor (cuma ngambil tiket sebelum dia churn)
df_tickets = pd.read_csv('support_tickets.csv')
df_accounts = pd.read_csv('customer_accounts.csv')

# Memperbaiki format tanggal
df_tickets['created_date'] = pd.to_datetime(df_tickets['created_date'], format='mixed', dayfirst=True)
df_accounts['unsubscribed_date'] = pd.to_datetime(df_accounts['unsubscribed_date'], format='mixed', dayfirst=True)

# Membuang tiket masa depan
df_tickets = df_tickets.merge(df_accounts[['customer_id', 'unsubscribed_date']], on='customer_id', how='left')
kondisi_valid = (df_tickets['created_date'] <= df_tickets['unsubscribed_date']) | (df_tickets['unsubscribed_date'].isna())
df_tickets_valid = df_tickets[kondisi_valid].copy()


# 2. bikin skor keparahan tiket (pengganti nlp textblob yang salah baca)
def hitung_skor_keparahan(row):
    skor = 0
    # cek prioritas
    if row['priority'] in ['High', 'Critical']:
        skor += 3  # masalah berat
    elif row['priority'] == 'Medium':
        skor += 2  # masalah sedang
    else:
        skor += 1  # masalah ringan (low)
        
    # cek status (kalau masih open berarti lebih gawat)
    if row['status'] == 'Open':
        skor += 2
        
    return skor

df_tickets_valid['severity_score'] = df_tickets_valid.apply(hitung_skor_keparahan, axis=1)


# 3. gabungin data per pelanggan (biar id nya ga dobel)
df_komplain = df_tickets_valid.groupby('customer_id').agg(
    total_ticket=('ticket_id', 'count'),
    avg_severity=('severity_score', 'mean'),
    # ngitung berapa banyak tiket yang skornya >= 4 (misal: high priority atau tiket medium tapi nganggur/open)
    severe_ticket_count=('severity_score', lambda x: (x >= 4).sum())
).reset_index()


# 4. bikin rasio tiket komplain parah (fitur andalan buat ml)
df_komplain['severe_ticket_ratio'] = df_komplain['severe_ticket_count'] / df_komplain['total_ticket']


# 5. SEGMENTASI TIPE PELANGGAN BERDASARKAN KEPARAHAN KOMPLAIN
def segmentasi_tipe_pelanggan(avg_severity, severe_ratio, total_tickets):
    # jika tidak pernah buat tiket, maka "loyal_quiet"
    if total_tickets == 0:
        return 'loyal_quiet'
    
    # jika rata-rata komplainnya parah banget & rasio tiket bahayanya tinggi -> "problematic"
    if avg_severity >= 4 or severe_ratio > 0.5:
        return 'problematic'
    
    # jika lumayan sering komplain atau ada tiket bahaya -> "at_risk"
    if avg_severity >= 3 or severe_ratio > 0.3:
        return 'at_risk'
    
    # komplain ringan dan biasanya cepat closed -> "satisfied"
    return 'satisfied'

df_komplain['customer_type'] = df_komplain.apply(
    lambda row: segmentasi_tipe_pelanggan(row['avg_severity'], row['severe_ticket_ratio'], row['total_ticket']),
    axis=1
)


# 6. gabungin balik ke master akun biar pelanggannya genap 3000
df_final = df_accounts[['customer_id']].merge(df_komplain, on='customer_id', how='left')


# 7. PERBAIKAN BUG FILLNA: isi kolom teks dulu, baru kolom angka
# pelanggan yang anteng diisi tipe 'loyal_quiet'
df_final['customer_type'] = df_final['customer_type'].fillna('loyal_quiet')
# sisanya (angka-angka metrik komplain) baru dinolin
df_final = df_final.fillna(0)


# save hasilnya
df_final.to_csv('hasil_komplain.csv', index=False)
print("✅ file hasil_komplain.csv sukses dibikin dan pelanggan genap 3000 siap dipake buat ml!")


# 8. TAMPILKAN ANALISIS TIPE PELANGGAN
print("\n" + "="*50)
print("📊 BREAKDOWN TIPE PELANGGAN BERDASARKAN KEPARAHAN TIKET")
print("="*50)

tipe_breakdown = df_final['customer_type'].value_counts()
print("\njumlah pelanggan per tipe:")
print(tipe_breakdown)

print("\npersentase per tipe:")
print((tipe_breakdown / len(df_final) * 100).round(2))

print("\nringkasan setiap tipe:")
print("- loyal_quiet    : tidak pernah komplain, anteng")
print("- satisfied      : komplain ringan dan biasanya cepat diselesaikan")
print("- at_risk        : lumayan sering komplain atau ada tiket yang belum diselesaikan")
print("- problematic    : komplainnya gawat (high/critical) dan banyak yang nyangkut (open)")

✅ file hasil_komplain.csv sukses dibikin dan pelanggan genap 3000 siap dipake buat ml!

📊 BREAKDOWN TIPE PELANGGAN BERDASARKAN KEPARAHAN TIKET

jumlah pelanggan per tipe:
customer_type
satisfied      2525
at_risk         319
loyal_quiet     121
problematic      35
Name: count, dtype: int64

persentase per tipe:
customer_type
satisfied      84.17
at_risk        10.63
loyal_quiet     4.03
problematic     1.17
Name: count, dtype: float64

ringkasan setiap tipe:
- loyal_quiet    : tidak pernah komplain, anteng
- satisfied      : komplain ringan dan biasanya cepat diselesaikan
- at_risk        : lumayan sering komplain atau ada tiket yang belum diselesaikan
- problematic    : komplainnya gawat (high/critical) dan banyak yang nyangkut (open)
